In this notebook we're going to have a practical code oriented look at understanding the Transformer architecture from the [Attention is All you Need](https://proceedings.neurips.cc/paper/2017/hash/3f5ee243547dee91fbd053c1c4a845aa-Abstract.html) paper. There are already very good articles on this such as [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/), albeit for pytorch, this  is going to be in jax and related libraries. 

In [8]:
import jax
from jax import numpy as jnp
import numpy as np
import flax.linen as nn

## What is Transformer? 

Transformer is an Encoder-Decoder based neural network model architecture, which processes sequences. One class of its predecessors, recurrent networks, e.g. [RNN](https://www.osti.gov/biblio/6910294) and [LSTM](https://dl.acm.org/doi/10.1162/neco.1997.9.8.1735) were called seq2seq or sequence to sequence networked because they could take one sequence $X$ as input and map it to another sequence (the target) $Y$. In a typical sequence to sequence matching task, both the input and the target sequence can be of different lengths. Meaning, they do not have be a 1:1 match in size. So, $X$ can have $n$ elements and $Y$ can have $m$ elements. 

Seq2Seq Models are very popular and are also very effective in applications such as Speech Recognition (speech to text) and Synthesis (text to speech), Machine Translation (Dutch to English) etc. There is another textbook term for these models, sequence transduction models. Before transformers, successful (at their respective tasks) sequence transduction models were either based on recurrent or [convolution based architectures](https://aclanthology.org/W18-5408/). 

### Attention



When we are mapping one sequence to another, we are actually looking for some dependency in between the elements of the sequences. In sequence transduction, this dependency is learnt as a function. It can have different names based on the problem domain, e.g. alignment function in machine translation. Question is, how do we define this function? 

[Attention](https://iclr.cc/archive/www/lib/exe/fetch.php%3Fmedia=iclr2015:bahdanau-iclr2015.pdf) is one such dependency function which creates a learnt dependency representation from two sequences. Mathematically defined, a general form of attention would look like this.

We have a source sequence $X = <x_1, x_2, ..... x_n >$ and a target sequence $Y = <y_1, y_2, .... y_n >$, where $n$ and $m$ are their respective lengths. Through attention function $A(X, Y)$ we would like to know the probability of an element in $X$ having a dependency with another element in $Y$. So attention gives us a probability based representation of the depdendencies between the elements of two sequences. 

Let's try to understand attention from a more general perspective. What we are asking is that we would like to search for a dependency function between two sequences but we do not have the information on the search space. Instead, we have the two sequences. So what can we do to accomplish this search? In other words, we know the purpose of the search space and what kind of information it should contain but not the contents of it. 

Now we formalise this search with 3 terms. Query($Q$), Key($K$) and Values($V$). In simple terms, query is, our search query, or what we're looking for. Keys are information which are available to us, and values are the content that gets retrieved and aggregated when a key is selected. Now earlier, we said that we do not know the contents of the search space or this setup. Then how do we know what keys we have for a query? The answer is that we learn them, or give them some parameters which we train on some task specific data.

Perhaps we can explain this easier with a practical example. Let's say that we would like to know the dependencies between the words of these two sentences, one in German, the other in English.

- German: Der Flughafen ist nur einen Katzensprung von hier entfernt.
- English: The airport is only at a stone's throw from here.

Now, computers don't understand raw text, which otherwise would've saved us a lot of work building string and text processing algorithms, so we create some representation from the word using numbers and some function which the computers can comprehend. We will come back to these functions a bit later when we describe Transformer in detail but for now, we need to focus on attention. 

Let's say that German is our source sequence and English the target one, and we denote them again, with $X$ and $Y$ and let's assume that $X$ and $Y$ already have some numeric representation of these two sentences. What we now would like to know is, what are the dependencies between the German words and the English words. And our search query for the two sequence is:

```bash
If I was given the words from this English sentence, how would I know which German words they relate to? 
```

In terms of the formalism we defined above, our query comes from the target words, and we would be looking for keys in the source sequence. What about the values then? You might have noticed by now that this is more of a translation task, and we're looking for the alignment, hence, values will come from the German words to carry the information that gets aggregated.

Like we said, we will need to parameterise $Q$, $K$, and $V$. So from $Y$ (the target, English), we first create $Q$ via a linear projection.
$$Q = Linear(Y)$$

Now since keys come from $X$ (the source, German)
$$K = Linear(X)$$

And values?
$$V = Linear(X)$$

Now that we have $Q$ and $K$, we take their dot products and get the keys relevant to the query. Perhaps this will help to understand why dot product works at retrieving the relevant information. 

Suppose our query looked like $[1, 2, 3, 89, 100]$ and our keys as this: $[1, 1, 1, 0, 0]$. If we read the query again, it's saying that the last two elements are 0. If we take the dot product, the last two elements in the result will be 0. When retrieving relevant information, this is a very effective yet simple technique to 0 out what you don't need. Now our keys come from a linear projection so we can't deliberately 0 out words, if we could, we already knew the dependency function!


The dot product is our attention scores, $a = QK^T$. Applying softmax to the attention scores, we get a probability distribution over the scores, $\alpha = softmax(a)$. A probability distribution sums to 1 and the values are in between 0 and 1 (inclusive). Then what do with do with $V$? We retrieve again with dot product, which gives us the attention representation, $A = \alpha V$. 

So attention, in actual is a probability weighted aggregation of search information. We learn this over multiple iterations when training a model. 


The version of attention we just described is called the dot attention. There's another version, which is additive and replaces the dot product with a feed forward neural network. Since Transformer in the paper used scaled dot attention, we would skip the other version. (What is the scaling part? We'll get back to it in a while.)

Now a code example. 

In [9]:
n = 64
m = 64

X = jax.random.normal(jax.random.PRNGKey(0), (1, n))
Y = jax.random.normal(jax.random.PRNGKey(1), (1, m))

# batch the inputs to have a batch dimension of 1
X_batched = jnp.expand_dims(X, axis=0)  # (1, seq, d)
Y_batched = jnp.expand_dims(Y, axis=0)  # (1, seq, d)

In [10]:
# some arbitrary number of features for the dense layers for linear projections of Q, K, and V. 
n_features = 64 

class DotAttention(nn.Module):
    features: int

    @nn.compact
    def __call__(self, source, target):
        # Linear projections
        Q = nn.Dense(features=self.features)(target)
        K = nn.Dense(features=self.features)(source)
        V = nn.Dense(features=self.features)(source)

        # Compute attention scores
        # Q: (B, seq_q, d), K: (B, seq_k, d)
        scores = jnp.einsum('btd,bkd->btk', Q, K)  # (seq_q, seq_k)

        # Apply softmax to get attention weights
        weights = jax.nn.softmax(scores, axis=-1)  # (seq_q, seq_k)

        # Compute the weighted sum of values
        output = jnp.einsum('btk,bkd->btd', weights, V)  # (seq_q, d_v)

        return output

# -------------
attention = DotAttention(features=n_features)
attention_params = attention.init(jax.random.PRNGKey(2), X_batched, Y_batched)
attention_output = attention.apply(attention_params, X_batched, Y_batched)


assert attention_output.shape == (1, 1, n_features), f"Expected output shape (1, 1, {n_features}), but got {attention_output.shape}"

## Self Attention

Transformer uses self attention. It's still attention, except that instead of separate source and target sequences, we learn the dependencies between the elements of a single sequence, against itself. So the source and target are the same sequences. Code wise, the definition remains the same, instead of `(X, Y)`, you'll invoke attention with `(X, X)`.

A good question here would be that the transformer in the paper is aimed at machine translation, so why not use the general attention method instead of self attention. What are the benefits of learning the intra-sequence dependencies from the same sequence?

In the paper, the authors do mention this and say that in practice, self attention lets each element in the sequence attend to every other element, which means every element's representation already incorporates the whole sequence. This is what the encoder does: it turns the source sequence into a sequence of contextualized representations, with no recurrence and no long-range path length. Concretely, a self-attention layer has $O(n²·d)$ complexity per layer, but only $O(1)$ sequential operations and a maximum path length of $O(1)$ — meaning any two elements can be connected in a constant number of steps, with all elements computed in parallel. Compare this to a recurrent layer, which needs $O(n)$ sequential operations and has an $O(n)$ path length between any two elements. In practice, this means the model can learn dependencies without regard to their distance, and the decoder's self-attention layer, with a causal mask, enforces the auto-regressive property, so each element can only attend to the elements up to and including itself.

This table from the paper gives a detailed comparison on the complexity of the different approaches compared to self attention.

![cost](../images/transformers/cost.png)


## Scaled Dot Product Attention
The inputs to self attention are of the dimension $d_k$ for query and keys, and for values it's $d_v$. The dot product of query and keys is divided by $\sqrt{d_k}$ before softmax is applied. The authors hypothesise that the dot products get very large in magnitude for large values of $d_k$, pushing the softmax into regions where it has extremely small gradients. The scaling counteracts this.

In [11]:
class ScaledDotAttention(nn.Module):
    features: int

    @nn.compact
    def __call__(self, source, target):
        # Linear projections
        Q = nn.Dense(features=self.features)(target)
        K = nn.Dense(features=self.features)(source)
        V = nn.Dense(features=self.features)(source)

        # Compute attention scores
        # Q: (B, seq_q, d), K: (B, seq_k, d)
        scores = jnp.einsum('btd,bkd->btk', Q, K)  # (B, seq_q, seq_k)

        # Scale the scores by the square root of the feature dimension
        scale_factor = jnp.sqrt(self.features)
        scaled_scores = scores / scale_factor

        # Apply softmax to get attention weights
        weights = jax.nn.softmax(scaled_scores, axis=-1)  # (B, seq_q, seq_k)

        # Compute the weighted sum of values
        output = jnp.einsum('btk,bkd->btd', weights, V)  # (B, seq_q, d_v)

        return output
    
    
# -------------
scaled_attention = ScaledDotAttention(features=n_features)
scaled_attention_params = scaled_attention.init(jax.random.PRNGKey(3), X_batched, X_batched)
scaled_attention_output = scaled_attention.apply(scaled_attention_params, X_batched, X_batched) 
assert scaled_attention_output.shape == (1, 1, n_features), f"Expected output shape (1, 1, {n_features}), but got {scaled_attention_output.shape}"

## Self Attention Heads and Multiples of them

As we know by now, self attention provides us with a representation for each element (or, position in the paper) in a sequence. However, that's just one set of representations. In the paper the authors state that instead of one such attention layer, performing several attention layers in parallel,each with its own learned linear projections. This provides much more benefit, as each of them can learn [different aspects from the sequence](https://aclanthology.org/2020.findings-emnlp.389/). Each of these attention layers is thus called a head, and together this collection of attention heads is called multi head attention. Another benefit of multiple heads is that each head can run in parallel, and the total computational cost is similar to that of a single full-dimensional head, since each head works with a reduced dimension. In comparison, in recurrent networks, the elements are processed in a sequential manner. There is a fixed-size state that carries information from one position to the next and this recurrence makes it impossible to have parallelisation.


![multi-head-attention](../images/transformers/mha.png)


With this we have covered the most important component of the transformer architecture. Now we can look at the overall architecture and understand how each component fits in.

![progress_one](../images/transformers/progress_1.png)

In [12]:
class MultiHeadAttention(nn.Module):
    num_heads: int
    features: int

    @nn.compact
    def __call__(self, source, target):
        # Split the features into multiple heads
        head_dim = self.features // self.num_heads
        Q = nn.Dense(features=self.features)(target)
        K = nn.Dense(features=self.features)(source)
        V = nn.Dense(features=self.features)(source)

        # Reshape for multi-head attention
        Q = Q.reshape(Q.shape[0], Q.shape[1], self.num_heads, head_dim)  # (B, seq_q, num_heads, head_dim)
        K = K.reshape(K.shape[0], K.shape[1], self.num_heads, head_dim)  # (B, seq_k, num_heads, head_dim)
        V = V.reshape(V.shape[0], V.shape[1], self.num_heads, head_dim)  # (B, seq_k, num_heads, head_dim)

        # Compute attention scores
        # Q: (B, seq_q, num_heads, head_dim), K: (B, seq_k, num_heads, head_dim)
        # we need per h bathed (b) qk product
        scores = jnp.einsum('bqhd,bkhd->bhqk', Q, K)  # (B, num_heads, seq_q, seq_k)

        # Scale the scores by the square root of the head dimension
        scale_factor = jnp.sqrt(head_dim)
        scaled_scores = scores / scale_factor

        # Apply softmax to get attention weights
        weights = jax.nn.softmax(scaled_scores, axis=-1)  # (B, num_heads, seq_q, seq_k)

        # Compute the weighted sum of values
        # weights: (B, num_heads, seq_q, seq_k), V: (B, seq_k, num_heads, head_dim)
        output = jnp.einsum('bhqk,bkhd->bqhd', weights, V)  # (B, seq_q, num_heads, head_dim)

        # Concatenate heads and project to output features
        output = output.reshape(output.shape[0], output.shape[1], self.features)  # (B, seq_q, features)
        output = nn.Dense(features=self.features)(output)

        return output
    
# -------------
num_heads = 8
multi_head_attention = MultiHeadAttention(num_heads=num_heads, features=n_features)
multi_head_attention_params = multi_head_attention.init(jax.random.PRNGKey(4), X_batched, X_batched)
multi_head_attention_output = multi_head_attention.apply(multi_head_attention_params, X_batched, X_batched)
assert multi_head_attention_output.shape == (1, 1, n_features), f"Expected output shape (1, 1, {n_features}), but got {multi_head_attention_output.shape}"

## Positionwise Feed Forward Network

Let's recap a bit. Transformer uses self attention, which does not have recurrence. Alongside, there are multiple such attention layers which act in parallel to form multiple attention heads. From these attention heads we get elementwise representations for the input sequence, which are concatenated in the end to form one aggregated elementwise representation. (Hence the 'add' term in the architecture diagram in the paper.) The attention representation we thus get is an weighted average over the values. 

However, there are two important caveats we have to note here. One, each attention head is independent. Even after the aggregation over the values, each element or position in a sequence has a representation where the element knows how it relates to the other elements but that information is not shared in between the elements of the sequence. Two, since this is an average operation, whatever complex dependency we expect the heads to capture will now have a kind of collapsed or compressed representation. As a result, we would be missing out on very intricate dependencies. If we recall, learning complex and intricate dependencies is why we are using self attention or attention. 

To get around these caveats, we need a transform function, which will create a per element representation and also preserve the complex dependencies averaged over from the multi head attention operations. Another thing to note here is that the attention heads do no share any parameters in between them, since they're independent. This also makes the representations independent of each other. Let's think of this from a semantic perspective. Let's say that you're reading a sentence. Sentences are a sequence of words. After reading all the words, you now have an understanding of the meaning of it. If all the words had very independent meaning and added nothing to the overall "context" of the sentences, you won't probably understand what it meant. The same analogy applies here. We are creating a intra-sequence dependency to get some semantic notion of the sequence. Without a shared space for the meaning, we might end up with incohesive sequences or a wrong understanding of it. As such, this transorm function also needs to create representations from a shared meaning space. In neural network terms, this means that this transform function needs to have a shared parameter space. We use feed forward networks with a non linear activation, ReLU to create this transform function.


From the paper itself:

![ffn](../images/transformers/ffn.png)


In [13]:
class PositionwiseFeedForward(nn.Module):
    features: int
    hidden_features: int

    @nn.compact
    def __call__(self, x):
        # First linear layer
        x = nn.Dense(features=self.hidden_features)(x)
        x = nn.relu(x)
        # Second linear layer
        x = nn.Dense(features=self.features)(x)
        return x
    
# -------------
hidden_features = 128
positionwise_feedforward = PositionwiseFeedForward(features=n_features, hidden_features=hidden_features)
positionwise_feedforward_params = positionwise_feedforward.init(jax.random.PRNGKey(5), X_batched)
positionwise_feedforward_output = positionwise_feedforward.apply(positionwise_feedforward_params, X_batched)
assert positionwise_feedforward_output.shape == (1, 1, n_features), f"Expected output shape (1, 1, {n_features}), but got {positionwise_feedforward_output.shape}"

## What is Add & Norm?

**Add & Norm** in the Transformer architecture (diagram above) refers to [residual connections](https://arxiv.org/abs/1512.03385) and [layer normalisation](https://arxiv.org/abs/1607.06450). For simplicity, we will skip the full derivation of both mechanisms and focus on their purpose in the Transformer.

In each block, an Add & Norm operation wraps both sublayers, the multi-head attention operation and the feed-forward network (FFN). Given a sublayer input $x$ and its transformation $\text{Sublayer}(x)$, the layer computes:

$$\text{LayerNorm}(x + \text{Sublayer}(x))$$

Each component addresses a distinct training challenge:

* **Add (Residual connection):** Rather than replacing $x$ with $\text{Sublayer}(x)$, the residual connection performs an element-wise addition of the original input and its transformed output. In the forward pass, this allows each sublayer to learn an incremental update instead of reconstructing the representation from scratch. In the backward pass, it preserves a direct path for gradients to flow to earlier layers, mitigating vanishing gradients in deep networks.
* **Norm (Layer normalisation):** Repeatedly summing sublayer outputs causes activation magnitudes and variance to grow with depth. Layer normalisation rescales activations across the feature dimension for each token independently, keeping values bounded and stabilising training without depending on batch size or sequence length.



In [14]:
class AddNorm(nn.Module):
    features: int

    @nn.compact
    def __call__(self, x, sublayer_output):
        # Add the input and the sublayer output
        x = x + sublayer_output
        # Apply layer normalization
        x = nn.LayerNorm()(x)
        return x
    
# -------------
add_norm = AddNorm(features=n_features)
add_norm_params = add_norm.init(jax.random.PRNGKey(6), X_batched, positionwise_feedforward_output)
add_norm_output = add_norm.apply(add_norm_params, X_batched, positionwise_feedforward_output)
assert add_norm_output.shape == (1, 1, n_features), f"Expected output shape (1, 1, {n_features}), but got {add_norm_output.shape}"